# Week 33, Day 1: From vesicle to conductance: synaptic kinetics and receptor types

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w033/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Four receptors, four time courses

Write `dexp(t, tr, td)`, the difference of exponentials normalised to a peak of 1. Then compute, for each receptor in `kinetics`, its peak time and the area under its normalised waveform, `charge[name]` (ms), which sets how much current it carries for the same peak.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def dexp(t, tr, td):
    tp = tr * td / (td - tr) * np.log(td / tr)
    norm = 1 / (np.exp(-tp / td) - np.exp(-tp / tr))
    return ...

kinetics = {"AMPA": (0.2, 2.0), "NMDA": (2.0, 80.0), "GABA-A": (0.5, 8.0), "GABA-B": (30.0, 200.0)}
t = np.arange(0, 800, 0.01)
charge = {}
for name, (tr, td) in kinetics.items():
    g = dexp(t, tr, td)
    charge[name] = ...
    print(f"{name:7s}: peak at {t[np.argmax(g)]:6.2f} ms, area {charge[name]:7.1f} ms")
    plt.plot(t, g, label=name)
plt.xscale("log"); plt.xlim(0.05, 800); plt.xlabel("ms (log scale)"); plt.ylabel("g / g_max"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "dexp has a peak of 1 at the time given by the formula"
    t = np.arange(0, 50, 0.001); g = dexp(t, 0.5, 8.0)
    tp = 0.5 * 8 / 7.5 * np.log(16)
    assert abs(g.max() - 1) < 1e-6 and abs(t[np.argmax(g)] - tp) < 0.002

def test_charge():
    "charge holds the integral of each normalised waveform, and NMDA's is more than 20 times AMPA's"
    assert charge["NMDA"] > 20 * charge["AMPA"]

test_peak()
test_charge()
print("All checks passed")

### Exercise: Saturation at high rates

Complete `kinetic(times)`: Euler integration (dt = 0.01 ms) of the two-state model with α = 1.1 per mM per ms, β = 0.19 per ms and T = 1 mM during the 1 ms after each release time. Then compare its largest r for 10 releases at 20 and 500 Hz with the linear prediction, the sum of shifted copies of the single-release response.

In [ ]:
alpha, beta, dt = 1.1, 0.19, 0.01
t = np.arange(0, 200, dt)

def kinetic(times):
    T = np.zeros_like(t)
    for s in times:
        T[(t >= s) & (t < s + 1.0)] = 1.0
    r = np.zeros_like(t)
    for k in range(1, len(t)):
        r[k] = ...
    return r

single = kinetic([5.0])
single_peak = single.max()
peaks = {}
for f in (20, 500):
    times = 5.0 + np.arange(10) * 1000 / f
    linear = sum(np.interp(t - (s - 5.0), t, single, left=0) for s in times)
    peaks[f] = (kinetic(times).max(), linear.max())
    print(f"{f:3d} Hz: kinetic peak {peaks[f][0]:.3f}, linear sum {peaks[f][1]:.3f}")
print(f"single release opens {single_peak:.2f} of the receptors")

In [ ]:
import numpy as np
def test_single():
    "a single 1 ms pulse opens a fraction of receptors between 0.5 and 0.8"
    assert 0.5 < single_peak < 0.8

def test_saturation():
    "at 20 Hz the kinetic and linear predictions agree within 5%, at 500 Hz the kinetic peak is less than half the linear one"
    assert abs(peaks[20][0] / peaks[20][1] - 1) < 0.05 and peaks[500][0] < 0.5 * peaks[500][1]

test_single()
test_saturation()
print("All checks passed")

### Exercise: AMPA against NMDA summation

Drive a passive membrane (C = 200 pF, gL = 10 nS, rest -70 mV) with five synaptic conductances 50 ms apart (reversal 0 mV, peak 1 nS, Mg²⁺ block ignored). For each receptor, compute the depolarisation reached after the fifth input divided by that after the first, `ratio[name]`. The first peak is the maximum before the second input.

In [ ]:
def psp_train(tr, td, isi=50.0, n_in=5, gmax=1.0, T=500.0, dt=0.05):
    tt = np.arange(0, T, dt)
    g = gmax * sum(dexp(np.clip(tt - s, 0, None), tr, td) * (tt >= s) for s in 10 + isi * np.arange(n_in))
    V = np.empty_like(tt); v = -70.0
    for k in range(len(tt)):
        v += dt * (-10 * (v + 70) - g[k] * v) / 200.0
        V[k] = v
    return tt, V + 70

ratio = {}
for name in ("AMPA", "NMDA"):
    tt, dV = psp_train(*kinetics[name])
    first = dV[tt < 60].max()
    fifth = ...
    ratio[name] = fifth / first
    print(f"{name}: first {first:.2f} mV, after the fifth {fifth:.2f} mV, ratio {ratio[name]:.2f}")
    plt.plot(tt, dV, label=name)
plt.xlabel("ms"); plt.ylabel("depolarisation (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_ampa():
    "with AMPA, the fifth response is less than 1.3 times the first"
    assert ratio["AMPA"] < 1.3

def test_nmda():
    "with NMDA, the fifth response is more than twice the first"
    assert ratio["NMDA"] > 2

test_ampa()
test_nmda()
print("All checks passed")